## 1. Instalasi Dependency


In [ ]:
!pip install -q bertopic hdbscan umap-learn

## 2. Setup Konfigurasi dan Library


In [ ]:
import os, re, json, random, warnings, time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED         = 42
CPU_COUNT    = os.cpu_count() or 2
WORKERS      = max(1, CPU_COUNT - 1)
SEARCH_DIRS  = [Path("/kaggle/input"), Path("/kaggle/working")]
OUTPUT_DIR   = Path("/kaggle/working/outputs_scenario2")
KMEANS_SWEEP = list(range(10, 101, 5))

random.seed(SEED)
np.random.seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for env in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ[env] = str(CPU_COUNT)

print(f"CPU={CPU_COUNT} | WORKERS={WORKERS}")

## 3. Load Semua Output Skenario 1


In [ ]:
def find_file(filename):
    for base in SEARCH_DIRS:
        if not base.exists(): continue
        m = list(base.rglob(filename))
        if m: return m[0]
    return None

win_path = find_file("scenario1_winner_for_scenario2.csv")
if win_path:
    winner_s1 = pd.read_csv(win_path).iloc[0]
    WIN_EMB   = str(winner_s1["embedding_model"])
    print(f"Winner S1: {WIN_EMB} | K={int(winner_s1['K'])} | Cv={winner_s1['cv']} | F1={winner_s1['hungarian_f1']}")
else:
    WIN_EMB = "PubMedBERT"
    print("WARNING: winner CSV tidak ditemukan -> fallback PubMedBERT")

EMB_LABEL = WIN_EMB.lower()

emb_path     = find_file(f"embeddings_{EMB_LABEL}_scenario1.npy")
ds_path      = find_file(f"bertopic_{EMB_LABEL}_scenario1_dataset_used.csv")
reduced_path = find_file(f"reduced_umap_{EMB_LABEL}_scenario1.npy")

if not emb_path or not ds_path:
    raise FileNotFoundError(
        f"embeddings_{EMB_LABEL}_scenario1.npy atau "
        f"bertopic_{EMB_LABEL}_scenario1_dataset_used.csv tidak ditemukan."
    )

embeddings = np.load(emb_path).astype(np.float32)
data       = pd.read_csv(ds_path)
docs       = data["doc"].tolist()
y_true     = data["primary_label"].tolist()

assert embeddings.shape[0] == len(docs), "Embedding & docs tidak sejajar!"
print(f"Embeddings: {embeddings.shape} | Docs: {len(docs)} | Labels: {pd.Series(y_true).nunique()}")

## 4. Evaluation Setup


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import f1_score, silhouette_score
from scipy.optimize import linear_sum_assignment
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS

BIOMED_STOP = {
    "patient","patients","study","studies","result","results",
    "method","methods","conclusion","conclusions","background",
    "objective","objectives","data","analysis","significant",
    "significantly","associated","compared","comparison",
    "included","including","use","used","using","group","groups",
    "showed","shown"
}
STOP_ALL = set(STOPWORDS) | BIOMED_STOP

def tokenize(text):
    return [t for t in simple_preprocess(text, deacc=True, min_len=3) if t not in STOP_ALL]

with ProcessPoolExecutor(max_workers=WORKERS) as ex:
    tokenized_docs = list(ex.map(tokenize, docs, chunksize=500))

gensim_dict = Dictionary(tokenized_docs)
gensim_dict.filter_extremes(no_below=5, no_above=0.5, keep_n=50_000)

vectorizer_model = CountVectorizer(
    stop_words=list(ENGLISH_STOP_WORDS | BIOMED_STOP), ngram_range=(1, 2), min_df=1
)

def topic_uniqueness(tw):
    all_w = [w for t in tw for w in t]
    return round(len(set(all_w)) / len(all_w), 4) if all_w else 0.0

def hungarian_f1(y_true, y_topic):
    labels = sorted(set(y_true))
    topics = sorted(set(y_topic))
    l2i = {l: i for i, l in enumerate(labels)}
    t2i = {t: i for i, t in enumerate(topics)}
    yt  = np.array([l2i[y] for y in y_true])
    yp  = np.array([t2i[y] for y in y_topic])
    M   = np.zeros((len(topics), len(labels)))
    for t in topics:
        for l in labels:
            M[t2i[t], l2i[l]] = f1_score((yt == l2i[l]).astype(int),
                                          (yp == t2i[t]).astype(int), zero_division=0)
    ri, ci = linear_sum_assignment(-M)
    t2l = {topics[r]: labels[c] for r, c in zip(ri, ci)}
    fb  = pd.Series(y_true).mode()[0]
    return round(f1_score(y_true, [t2l.get(t, fb) for t in y_topic], average="macro", zero_division=0), 4)

def extract_topic_words(tm, topn=10):
    return [
        [w for w, _ in tm.get_topic(t)[:topn] if isinstance(w, str) and len(w) > 1]
        for t in sorted(x for x in tm.get_topics() if x != -1)
    ]

def coherence_cv(tw):
    if not tw: return 0.0
    return round(CoherenceModel(topics=tw, texts=tokenized_docs, dictionary=gensim_dict,
                                coherence="c_v", processes=WORKERS).get_coherence(), 4)

def safe_silhouette(X, labels):
    labels = np.array(labels)
    mask   = labels != -1
    if len(set(labels[mask])) < 2: return float("nan")
    return round(float(silhouette_score(X[mask], labels[mask])), 4)

print("Evaluation setup ready.")

## 5. UMAP: Load Cache dari S1 atau Run Ulang dengan n_jobs=1


In [ ]:
# n_jobs=1 wajib untuk determinisme — n_jobs>1 bisa menghasilkan hasil berbeda
# antar session meski random_state sama (thread ordering tidak guaranteed).
from umap import UMAP

if reduced_path:
    print(f"Loading cached UMAP reduced from Skenario 1: {reduced_path}")
    reduced = np.load(reduced_path).astype(np.float32)
else:
    print("Cache reduced_umap tidak ditemukan -> run UMAP ulang (n_jobs=1)...")
    reduced = UMAP(
        n_neighbors=15, n_components=5, min_dist=0.0,
        metric="cosine", random_state=SEED, n_jobs=1
    ).fit_transform(embeddings)
    # Simpan untuk run berikutnya
    np.save(OUTPUT_DIR / f"reduced_umap_{EMB_LABEL}_scenario1.npy", reduced)

print(f"Reduced: {reduced.shape}")

## 6. Helper: Run BERTopic dengan clusterer apapun


In [ ]:
from bertopic import BERTopic
from bertopic.dimensionality import BaseDimensionalityReduction

def run_arm(name, cluster_model, reduce_outliers=False):
    t0 = time.perf_counter()
    tm = BERTopic(
        umap_model=BaseDimensionalityReduction(),
        hdbscan_model=cluster_model,
        vectorizer_model=vectorizer_model,
        top_n_words=15, calculate_probabilities=False, verbose=False
    )
    topics, _ = tm.fit_transform(docs, embeddings=reduced)
    topics = list(topics)

    if reduce_outliers and -1 in topics:
        topics = tm.reduce_outliers(docs, topics)
        tm.update_topics(docs, topics=topics, vectorizer_model=vectorizer_model)

    fit_sec      = round(time.perf_counter() - t0, 3)
    k            = len(set(topics)) - (1 if -1 in topics else 0)
    outlier_rate = round(float(np.mean(np.array(topics) == -1)), 4)
    tw           = extract_topic_words(tm, topn=10)
    cv           = coherence_cv(tw)
    f1           = hungarian_f1(y_true, topics)
    hm           = round(2 * cv * f1 / (cv + f1), 4) if (cv + f1) > 0 else 0.0

    row = {
        "arm"             : name,
        "n_topics"        : k,
        "outlier_rate"    : outlier_rate,
        "cv"              : cv,
        "topic_uniqueness": topic_uniqueness(tw),
        "hungarian_f1"    : f1,
        "harmonic_mean"   : hm,
        "silhouette"      : safe_silhouette(reduced, topics),
        "fit_seconds"     : fit_sec,
    }
    print(f"{name} -> K={k} | Outlier={outlier_rate} | Cv={cv} | F1={f1} | HM={hm} | {fit_sec:.1f}s")
    return tm, topics, tw, row

print("Helper ready.")

## 7. Lengan A: HDBSCAN


In [ ]:
from hdbscan import HDBSCAN

mcs_path = find_file(f"bertopic_{EMB_LABEL}_mcs_sweep_results.csv")
MCS = int(pd.read_csv(mcs_path).sort_values("harmonic_mean_cv_f1", ascending=False).iloc[0]["min_cluster_size"]) if mcs_path else 60
print(f"MIN_CLUSTER_SIZE = {MCS} ({'dari sweep' if mcs_path else 'fallback default'})")

# Verifikasi: HDBSCAN standalone harus konsisten dengan S1
hdb_check = HDBSCAN(min_cluster_size=MCS, metric="euclidean",
                    cluster_selection_method="eom", core_dist_n_jobs=WORKERS)
k_check   = len(set(hdb_check.fit_predict(reduced))) - (1 if -1 in hdb_check.labels_ else 0)
print(f"Verifikasi K standalone = {k_check} (expected dari S1: {int(winner_s1['K']) if win_path else 'N/A'})")

hdb = HDBSCAN(min_cluster_size=MCS, metric="euclidean",
              cluster_selection_method="eom", prediction_data=True,
              gen_min_span_tree=True, core_dist_n_jobs=WORKERS)
tm_hdb, top_hdb, tw_hdb, row_hdb = run_arm("HDBSCAN_natural", hdb)
row_hdb["dbcv"] = round(float(getattr(hdb, "relative_validity_", float("nan"))), 4)
K_HDB = row_hdb["n_topics"]

hdb2 = HDBSCAN(min_cluster_size=MCS, metric="euclidean",
               cluster_selection_method="eom", prediction_data=True,
               gen_min_span_tree=True, core_dist_n_jobs=WORKERS)
tm_hdbr, top_hdbr, tw_hdbr, row_hdbr = run_arm("HDBSCAN_reduced_outliers", hdb2, reduce_outliers=True)
row_hdbr["dbcv"] = round(float(getattr(hdb2, "relative_validity_", float("nan"))), 4)

print(f"\nNatural K={K_HDB} | DBCV={row_hdb['dbcv']}")

## 8. Lengan B: K-Means — Silhouette Sweep


In [ ]:
from sklearn.cluster import KMeans

print("K-Means silhouette sweep...")
sweep = []
for k in KMEANS_SWEEP:
    lab = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit_predict(reduced)
    sil = round(float(silhouette_score(reduced, lab)), 4)
    sweep.append({"K": k, "silhouette": sil})
    print(f"  K={k} | Silhouette={sil}")

sweep_df = pd.DataFrame(sweep)
K_STAR   = int(sweep_df.sort_values("silhouette", ascending=False).iloc[0]["K"])
print(f"\nK* (silhouette-optimal) = {K_STAR}")

tm_km,  top_km,  tw_km,  row_km  = run_arm(f"KMeans_K{K_HDB}_matched",
    KMeans(n_clusters=K_HDB, random_state=SEED, n_init=10))
row_km["dbcv"] = float("nan")

tm_kms, top_kms, tw_kms, row_kms = run_arm(f"KMeans_K{K_STAR}_optimal",
    KMeans(n_clusters=K_STAR, random_state=SEED, n_init=10))
row_kms["dbcv"] = float("nan")

## 9. Gabung Hasil & Tentukan Pemenang


In [ ]:
COLS = ["arm","n_topics","outlier_rate","cv","topic_uniqueness",
        "hungarian_f1","harmonic_mean","silhouette","dbcv","fit_seconds"]

results = pd.DataFrame([row_hdb, row_hdbr, row_km, row_kms])
if "dbcv" not in results.columns: results["dbcv"] = float("nan")
results = results.reindex(columns=COLS)

print("=== Hasil Skenario 2 ===")
display(results)

head   = results[results["arm"].isin(["HDBSCAN_reduced_outliers", f"KMeans_K{K_HDB}_matched"])].copy()
winner = head.sort_values("harmonic_mean", ascending=False).iloc[0]

print("\n" + "="*50)
print(f"PEMENANG S2: {winner['arm']}")
print(f"  Cv={winner['cv']} | F1={winner['hungarian_f1']} | HM={winner['harmonic_mean']}")
print("="*50)

## 10. Save Output


In [ ]:
import pickle

results.to_csv(OUTPUT_DIR / "scenario2_results.csv", index=False)
sweep_df.to_csv(OUTPUT_DIR / "scenario2_kmeans_silhouette_sweep.csv", index=False)
pd.DataFrame([winner]).to_csv(OUTPUT_DIR / "scenario2_winner_for_scenario3.csv", index=False)

for nm, tw in [
    ("hdbscan_natural",          tw_hdb),
    ("hdbscan_reduced_outliers", tw_hdbr),
    (f"kmeans_k{K_HDB}_matched", tw_km),
    (f"kmeans_k{K_STAR}_optimal",tw_kms),
]:
    pd.DataFrame({
        "topic_id" : range(len(tw)),
        "top_words": [", ".join(t) for t in tw]
    }).to_csv(OUTPUT_DIR / f"scenario2_topics_{nm}.csv", index=False)

# Model terbaik + artifacts untuk Skenario 3 (XAI)
model_map = {
    "HDBSCAN_natural"              : (tm_hdb,  top_hdb),
    "HDBSCAN_reduced_outliers"     : (tm_hdbr, top_hdbr),
    f"KMeans_K{K_HDB}_matched"     : (tm_km,   top_km),
    f"KMeans_K{K_STAR}_optimal"    : (tm_kms,  top_kms),
}
best_tm, best_topics = model_map[winner["arm"]]

with open(OUTPUT_DIR / "scenario2_best_model.pkl", "wb") as f:
    pickle.dump(best_tm, f)
np.save(OUTPUT_DIR / "scenario2_best_topics.npy",  np.array(best_topics))
np.save(OUTPUT_DIR / "scenario2_reduced_umap.npy", reduced)

print(f"Saved to {OUTPUT_DIR}")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}")